<a href="https://colab.research.google.com/github/Santo29SL/CircularTraingle/blob/main/s2s_itr2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
print("PyTorch:", torch.__version__, "| CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - change runtime to T4")

PyTorch: 2.11.0+cu130 | CUDA: True
GPU: Tesla T4


In [ ]:
%%bash
cd /content
[ -d TripoSR ] || git clone https://github.com/VAST-AI-Research/TripoSR.git
cd TripoSR
pip install -q scikit-build-core pybind11 ninja
pip install -q --no-build-isolation --no-cache-dir git+https://github.com/tatsy/torchmcubes.git
pip install -q xatlas==0.0.10 omegaconf==2.3.0 einops==0.7.0 transformers==4.35.0 \
    trimesh moderngl==5.10.0 "rembg[cpu]" "imageio[ffmpeg]" opencv-python-headless \
    fastapi uvicorn python-multipart
# Pillow is intentionally NOT pinned to 10.1.0 (cannot build on Python 3.13)
# gradio is intentionally NOT installed (not needed, caused long resolver backtracking)
pip freeze > /content/requirements-frozen.txt

Cloning into 'TripoSR'...
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pytensor 2.38.3 requires numba<=0.65.1,>=0.58, but you have numba 0.68.1 which is incompatible.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.17.3 which is incompatible.
sentence-transformers 5.7.0 requires huggingface-hub>=0.23.0, but you have huggingface-hub 0.17.3 which is incompatible.
sentence-transformers 5.7.0 requires tokenizers>=0.19, but you have tokenizers 0.14.1 which is incompatible.
sentence-transformers 5.7.0 requires transformers<6.0.0,>=4.41.0, but you have transformers 4.35.0 which is incompatible.
diffusers 0.42.1 requires huggingface-hub<3.0,>=1.32.0, but you have huggingface-hub 0.17.3 which is incompatible.
gradio 6.30.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.17.3 which is incompatible.
gradio

In [ ]:
import sys, os
os.chdir("/content/TripoSR"); sys.path.insert(0, "/content/TripoSR")
import torch, torchmcubes, trimesh, xatlas, PIL, cv2, transformers
from tsr.system import TSR
print("OK | torch", torch.__version__, "| trimesh", trimesh.__version__,
      "| Pillow", PIL.__version__, "| transformers", transformers.__version__)

/usr/local/lib/python3.13/dist-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(


OK | torch 2.11.0+cu130 | trimesh 5.2.0 | Pillow 13.0.0 | transformers 4.35.0


In [ ]:
import subprocess, time, requests, sys, os
subprocess.run("pkill -f 'uvicorn api:app' || true", shell=True)
api_log = open("/content/api.log", "w")
api_proc = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "api:app", "--host", "0.0.0.0", "--port", "8000"],
    cwd="/content/TripoSR", stdout=api_log, stderr=subprocess.STDOUT)
for i in range(90):                      # up to ~3 min (first run downloads 1.7 GB)
    time.sleep(2)
    try:
        r = requests.get("http://127.0.0.1:8000/", timeout=2)
        if r.status_code == 200:
            print("API ready:", r.json()); break
    except Exception:
        pass
    if api_proc.poll() is not None:
        print(open("/content/api.log").read()[-3000:]); raise SystemExit("API crashed")
else:
    print(open("/content/api.log").read()[-3000:])

API ready: {'status': 'online', 'service': 'Sketch2Shape TripoSR API', 'model_loaded': True, 'device': 'cuda', 'gpu': 'Tesla T4'}


In [ ]:
import requests, time, json
from PIL import Image, ImageDraw

# synthetic line-drawing "house" on transparent-ish white canvas
im = Image.new("RGB", (512, 512), "white"); d = ImageDraw.Draw(im)
d.rectangle([140, 240, 372, 420], outline="black", width=6)          # walls
d.polygon([(120, 245), (256, 120), (392, 245)], outline="black", width=6)  # roof
d.rectangle([225, 320, 285, 420], outline="black", width=6)          # door
im.save("/content/house_sketch.png")

def test(path, mode):
    t = time.time()
    with open(path, "rb") as f:
        r = requests.post("http://127.0.0.1:8000/generate",
                          files={"image": (os.path.basename(path), f, "image/png")},
                          data={"prompt": "", "mode": mode}, timeout=300)
    print(path, mode, "->", r.status_code, f"{time.time()-t:.1f}s")
    print(json.dumps(r.json(), indent=2)[:900]); return r.json()

res_house = test("/content/house_sketch.png", "sketch")
res_chair = test("/content/TripoSR/examples/chair.png", "photo")

/content/house_sketch.png sketch -> 200 9.0s
{
  "success": true,
  "job_id": "7091bb2bb8984c4d8a2e58241d729095",
  "glb_url": "/outputs/7091bb2bb8984c4d8a2e58241d729095/model.glb",
  "obj_url": "/outputs/7091bb2bb8984c4d8a2e58241d729095/model.obj",
  "processed_input_url": "/outputs/7091bb2bb8984c4d8a2e58241d729095/processed_input.png",
  "vertices": 105085,
  "faces": 209812,
  "dimensions_xyz": [
    0.8592,
    1.0242,
    0.9344
  ],
  "watertight": false,
  "glb_kb": 4101.7,
  "obj_kb": 11195.2,
  "timings": {
    "preprocess_s": 0.1,
    "inference_s": 5.09,
    "mesh_extraction_s": 2.91,
    "export_s": 0.71,
    "total_s": 8.81
  },
  "mode": "sketch",
  "resolution": 256,
  "prompt_received": false,
  "prompt_used": false,
  "note": "TripoSR reconstructs from the image only; the text prompt is not used to condition geometry."
}
/content/TripoSR/examples/chair.png photo -> 200 4.8s
{
  "success": true,
  "job_id": "71c580ff12a74d9a978e890904ca6100",
  "glb_url": "/outputs/71c5

In [ ]:
import numpy as np, trimesh
from google.colab import files
def validate(job):
    p = f"/content/TripoSR/api_outputs/{job['job_id']}/model.glb"
    scene = trimesh.load(p)
    mesh = trimesh.util.concatenate(list(scene.geometry.values())) if hasattr(scene, "geometry") else scene
    print("GLB KB:", round(os.path.getsize(p)/1024, 1), "| V:", len(mesh.vertices), "| F:", len(mesh.faces),
          "| dims XYZ:", np.ptp(mesh.vertices, axis=0).round(3), "| watertight:", mesh.is_watertight)
    # after orientation the tallest axis of an upright object (chair/house) should be Y
validate(res_chair); validate(res_house)
files.download(f"/content/TripoSR/api_outputs/{res_house['job_id']}/model.glb")

GLB KB: 1636.1 | V: 41900 | F: 83662 | dims XYZ: [0.598 0.93  0.537] | watertight: False
GLB KB: 4101.7 | V: 105085 | F: 209812 | dims XYZ: [0.859 1.024 0.934] | watertight: False


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import subprocess, re, time, requests
from pathlib import Path
if not Path("/content/cloudflared").exists():
    subprocess.run("wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared && chmod +x /content/cloudflared", shell=True, check=True)
subprocess.run("pkill -f cloudflared || true", shell=True)
tlog = open("/content/cloudflared.log", "w")
tunnel = subprocess.Popen(["/content/cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"],
                          stdout=tlog, stderr=subprocess.STDOUT)
PUBLIC_URL = None
for _ in range(40):
    time.sleep(2)
    m = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", Path("/content/cloudflared.log").read_text())
    if m: PUBLIC_URL = m.group(0); break
print("PUBLIC URL:", PUBLIC_URL)
print("Health:", requests.get(PUBLIC_URL + "/", timeout=15).json())
print("Open your Space with:  https://santo29sl-sketch2shape.hf.space/?api=" + PUBLIC_URL)

PUBLIC URL: https://referrals-causing-masters-much.trycloudflare.com


ConnectionError: HTTPSConnectionPool(host='referrals-causing-masters-much.trycloudflare.com', port=443): Max retries exceeded with url: / (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x7a9a9620a0d0>: Failed to resolve 'referrals-causing-masters-much.trycloudflare.com' ([Errno -2] Name or service not known)"))

In [ ]:
import requests, time
for i in range(30):
    try:
        r = requests.get(PUBLIC_URL + "/", timeout=10)
        if r.status_code == 200:
            print("OK:", r.json())
            print("Open:  https://santo29sl-sketch2shape.hf.space/?api=" + PUBLIC_URL)
            break
    except Exception as e:
        print(i, "waiting for DNS...", type(e).__name__)
    time.sleep(5)

OK: {'status': 'online', 'service': 'Sketch2Shape TripoSR API', 'model_loaded': True, 'device': 'cuda', 'gpu': 'Tesla T4'}
Open:  https://santo29sl-sketch2shape.hf.space/?api=https://rapid-acute-relations-countries.trycloudflare.com


In [ ]:
!grep -n "allow_origins\|allow_origin_regex" /content/TripoSR/api.py
!curl -s -i -H "Origin: https://test.hf.space" https://rapid-acute-relations-countries.trycloudflare.com/ | head -15

86:    allow_origins=["*"],
HTTP/2 530 
date: Sat, 31 Oct 2026 12:42:56 GMT
content-type: text/html; charset=UTF-8
content-length: 7739
cf-ray: a532be1698dcf7d6-LAX
cache-control: private, max-age=0, no-store, no-cache, must-revalidate, post-check=0, pre-check=0
expires: Thu, 01 Jan 1970 00:00:01 GMT
retry-after: 120
server: cloudflare
referrer-policy: same-origin
x-frame-options: SAMEORIGIN
x-robots-tag: none
x-robots-tag: none

<!doctype html>


In [ ]:
import requests, time
print("Current tunnel:", PUBLIC_URL)
for i in range(30):
    try:
        r = requests.get(PUBLIC_URL + "/", headers={"Origin": "https://test.hf.space"}, timeout=10)
        if r.status_code == 200:
            print("OK:", r.json())
            print("CORS header:", r.headers.get("access-control-allow-origin"))
            print("Use this URL in the Space:", PUBLIC_URL)
            break
        print(i, "status", r.status_code)
    except Exception as e:
        print(i, "waiting...", type(e).__name__)
    time.sleep(5)

Current tunnel: https://referrals-causing-masters-much.trycloudflare.com
OK: {'status': 'online', 'service': 'Sketch2Shape TripoSR API', 'model_loaded': True, 'device': 'cuda', 'gpu': 'Tesla T4'}
CORS header: *
Use this URL in the Space: https://referrals-causing-masters-much.trycloudflare.com
